## 1. Загрузка

In [ ]:
from pathlib import Path
import re

import numpy as np
import pandas as pd

pd.set_option("display.max_colwidth", 200)
pd.set_option("display.width", 200)

DATA_DIR = Path("data")
OUT_DIR = DATA_DIR / "processed"
OUT_DIR.mkdir(parents=True, exist_ok=True)

train   = pd.read_parquet(DATA_DIR / "train.parquet")
queries = pd.read_parquet(DATA_DIR / "benchmark_queries.parquet")
items   = pd.read_parquet(DATA_DIR / "benchmark_items.parquet")

NUM_COLS = ["item_latitude", "item_longitude", "item_price", "item_rating",
            "item_rating_reviews_count", "item_is_phone_hidden", "item_is_message_forbidden",
            "search_is_delivery_search"]
for df in (train, queries, items):
    for col in NUM_COLS:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").astype("float64")

print("train:", train.shape, "| queries:", queries.shape, "| items:", items.shape)

train: (497673, 19) | queries: (2452, 6) | items: (189212, 14)


## 2. Дубли в train

В EDA нашлось ~30 тысяч строк, где совпадает всё: запрос, все его фильтры и выбранное объявление. Новой информации они не несут, искажают статистику

In [ ]:
Q_COLS = ["search_query", "search_location_id", "search_is_delivery_search",
          "search_infm_params_text", "search_category"]

before = len(train)
train = train.drop_duplicates(subset=Q_COLS + ["item_id"]).reset_index(drop=True)
print(f"Удалено дублей: {before - len(train)}, осталось строк: {len(train)}")

Удалено дублей: 30619, осталось строк: 467054


## 3. Оставляем в корпусе только услуги???????

проверка категории 114

In [ ]:
print(queries["search_category"].value_counts())

# Что ищут люди, если категория не 114
display(queries.loc[queries["search_category"] != 114, ["search_query", "search_category"]])

# Каких категорий объявления в корпусе, и сколько их
display(items["item_category_id"].value_counts())

search_category
114    2230
0       222
Name: count, dtype: int64


,search_query,search_category
16,автомобиль бу под выкуп под такси,0
24,установка реечного потолка в ванной,0
28,ищу спарринг партнера по теннису,0
40,экскурсия в чечню,0
46,репетитор по веб разработке,0
...,...,...
2354,грузовой бортовой машины,0
2362,регулировка окон schuco,0
2388,грузовики для переезда,0
2412,замок гаражный реечный,0


item_category_id
114    187336
19        357
112       141
40        122
33        103
10         98
20         84
25         82
36         74
111        74
39         72
24         70
83         68
27         57
30         56
97         44
21         36
88         34
82         32
42         29
81         24
85         22
9          18
29         18
32         17
84         16
116        14
34         12
106        11
26         11
101        10
102         9
11          9
99          8
28          8
38          5
14          5
23          4
98          4
94          4
105         3
96          3
87          3
90          2
93          1
31          1
89          1
Name: count, dtype: int64

In [ ]:
SERVICE_CATEGORY = 114

share_train = (train["item_category_id"] == SERVICE_CATEGORY).mean()
share_items = (items["item_category_id"] == SERVICE_CATEGORY).mean()
print(f"Доля выборов в train из категории {SERVICE_CATEGORY}: {share_train:.5f}  "
      f"(это потолок recall после фильтра)")
print(f"Доля корпуса в категории {SERVICE_CATEGORY}:          {share_items:.3f}")

print("\nЧто лежит в корпусе вне услуг (примеры):")
display(items.loc[items["item_category_id"] != SERVICE_CATEGORY,
                  ["item_category_id", "item_title_raw"]].sample(15, random_state=0))

Доля выборов в train из категории 114: 0.99997  (это потолок recall после фильтра)
Доля корпуса в категории 114:          0.990

Что лежит в корпусе вне услуг (примеры):


,item_category_id,item_title_raw
44150,19,Забор из металлического штакетника
189182,20,Шкаф на балкон на заказ
78795,112,"Яндекс Директ, настройка рекламы"
10010,42,Офис в центре Нижнего Новгорода 181.3 м²
182569,40,"Нержавеющий реактор, диссольвер 250 л"
45062,112,"Газорезчик, маляр, стропальщик"
55723,19,"Жби кольца,выгребные ямы,септики под ключ"
1996,19,Тротуарная плитка и бордюри
27509,24,"Квартира-студия, 45 м², 4 кровати"
5998,112,Инженер-конструктор


Если доля выборов из категории 114 близка к 1 (например, 0.9999), фильтр практически ничего не стоит по recall, а корпус заметно уменьшается, то применяем

In [ ]:
items_before = len(items)
#items = items[items["item_category_id"] == SERVICE_CATEGORY].reset_index(drop=True)
# Было принято новое решение :
# Не удаляем другие категории: у 9% benchmark-запросов категория 0 (поиск по всем категориям),
# и их ответы могут лежать вне услуг. Категорию учтём на этапе поиска.
print("Корпус оставляем целиком:", len(items))
print("Из них не-услуг:", (items["item_category_id"] != SERVICE_CATEGORY).sum())

print(f"Корпус: было {items_before}, стало {len(items)}")

Корпус оставляем целиком: 189212
Из них не-услуг: 1876
Корпус: было 189212, стало 189212


## 4. Чистка текста


In [ ]:
def clean_text(s) -> str:
    """Чистит текст, сохраняя его естественный вид (регистр не трогаем)."""
    if not isinstance(s, str):
        return ""
    s = s.replace("\\n", " ").replace("\\r", " ").replace("\\t", " ")  # буквальные "\n"
    s = re.sub(r"[\n\r\t]", " ", s)                                     # настоящие переносы
    s = s.replace("ё", "е").replace("Ё", "Е")
    s = re.sub(r"[^\w\s.,:;!?()/%+№-]", " ", s)   # всё, что не буква/цифра/базовая пунктуация
    s = s.replace("_", " ")
    s = re.sub(r"\s+", " ", s).strip()
    return s

# Проверим на примерах
for t in ["🚗 Автоподбор и выездная диагностика\\nв Нижнем Новгороде!!!",
          "Натяжные   потолки ★★★ ЁЛКИ",
          None]:
    print(repr(t), "->", repr(clean_text(t)))

'🚗 Автоподбор и выездная диагностика\\nв Нижнем Новгороде!!!' -> 'Автоподбор и выездная диагностика в Нижнем Новгороде!!!'
'Натяжные   потолки ★★★ ЁЛКИ' -> 'Натяжные потолки ЕЛКИ'
None -> ''


## 5. Разбираем параметры объявления

Поле `item_infm_params_text` — это склеенные в одну строку пары «название поля → значение», **без разделителей**:

```
Вид услуги Красота, здоровье Место оказания услуг Москва, ул. Ленина, 5 Тип стоимости за услугу Начальная цена ...
```

В нём есть очень полезные для поиска части («Вид услуги Красота, здоровье», «Тип услуги Маникюр, педикюр») и мусор: адрес, график работы в секундах, тип стоимости. Адрес особенно вреден: слово «Москва» есть в тысячах объявлений, и по запросу «эвакуатор москва» BM25 может поднять наверх любое московское объявление

In [ ]:
PARAM_KEYS = [
    "Вид услуги", "Тип услуги", "Тип услуги автосервиса", "Название услуги",
    "Место оказания услуг", "Место сделки",
    "Тип стоимости", "Начальная цена", "Стоимость",
    "График работы от", "График работы до",
    "Работаете с юрлицами и ИП", "Работа по договору", "Опыт работы", "Гарантия",
    "Дополнительно", "Марка", "Модель",
    "Вид товара", "Тип товара", "Вид объявления", "Вид запчасти",
    "Онлайн-запись", "Рейтинг пользователя",
    # новые, найденные по длинным адресам:
    "Груз", "Тип техники", "Состояние", "Качество", "Направление", #"Доставка",
    "Вид торгового оборудования", "Чем вы занимаетесь", "Расстояние доставки",
    "Куда выезжаете", "Как вы работаете",
]
# Длинные ключи ставим первыми: иначе «Тип услуги автосервиса» распознается как «Тип услуги»
_keys_sorted = sorted(PARAM_KEYS, key=len, reverse=True)
KEY_RE = re.compile(r"(?<!\w)(" + "|".join(map(re.escape, _keys_sorted)) + r")(?!\w)")

ADDRESS_KEYS = {"Место оказания услуг", "Место сделки"}
DROP_KEYS = ADDRESS_KEYS | {"Тип стоимости", "Начальная цена", "Стоимость",
                            "График работы от", "График работы до"}
MAX_ADDRESS_WORDS = 12  # страховка: если адрес «съел» неизвестное поле, обрежем


def parse_params(raw) -> dict:
    """Разбирает строку параметров на поля."""
    text = clean_text(raw)
    parts = KEY_RE.split(text)          # ['текст до ключей', ключ1, значение1, ключ2, значение2, ...]
    head = parts[0].strip()
    pairs = [(k, v.strip()) for k, v in zip(parts[1::2], parts[2::2])]

    vid = [v for k, v in pairs if k == "Вид услуги" and v]
    tip = [v for k, v in pairs if k.startswith("Тип услуги") and v]
    addr = [v for k, v in pairs if k in ADDRESS_KEYS and v]
    addr = " ".join(addr[0].split()[:MAX_ADDRESS_WORDS]) if addr else ""

    # «Полезные» параметры: всё, кроме адреса, цен и графика
    useful = [head] + [f"{k} {v}" for k, v in pairs if k not in DROP_KEYS]
    return {
        "vid": vid[0].lower() if vid else "",
        "tip": " ; ".join(tip).lower(),
        "address": addr.lower(),
        "params_clean": re.sub(r"\s+", " ", " ".join(useful)).strip().lower(),
    }

example = ("Вид услуги Автосервис, аренда Тип услуги автосервиса Диагностика и ремонт авто "
           "Место оказания услуг Нижний Новгород, Советский район Тип стоимости за услугу "
           "Начальная цена Гарантия Опыт работы 10 лет и больше")
parse_params(example)

{'vid': 'автосервис, аренда',
 'tip': 'диагностика и ремонт авто',
 'address': 'нижний новгород, советский район',
 'params_clean': 'вид услуги автосервис, аренда тип услуги автосервиса диагностика и ремонт авто гарантия опыт работы 10 лет и больше'}

In [ ]:
from collections import Counter

cnt = Counter()
sample = items["item_infm_params_text"].dropna().sample(20000, random_state=0)
for t in sample:
    # слово с заглавной буквы + до двух следующих слов со строчной
    cnt.update(re.findall(r"(?<!\w)[А-ЯЁ][а-яё-]+(?: [а-яё-]+){0,2}", clean_text(t)))

for phrase, n in cnt.most_common(200):
    if not any(phrase.startswith(k) for k in PARAM_KEYS):
        print(n, phrase)

76570 Услуга
37034 Время для связи
32754 Продолжительность
31089 График работы
20757 Рабочие дни
19111 Своя услуга
18275 Время работы
10876 Производители
6586 Работаете с юрлицами
5580 Бригада
5410 Мероприятия
5310 По всему городу
4007 Готов закупить материалы
3773 Услуги
3718 Удаленно
3654 Выполняю заказы от
3483 Предоплата
3330 Специальность
3280 Женщина
3222 Год окончания
3123 Вторник
3123 Среда
3122 Четверг
3121 Пятница
3112 Понедельник
3047 Учебное учреждение
3034 Берете ли срочные
2994 Ваши клиенты
2916 Дни ср
2915 Дни вт
2912 Дни пн
2911 Дни чт
2905 Дни пт
2755 Занятия
2724 Суббота
2606 Аудитория
2570 Где вы оказываете
2525 Дни сб
2463 Автосервис
2437 Воскресенье
2281 Кто оказывает услуги
2271 Признак предзаполнения прайс
2254 Обучение
2225 Дни вс
2157 Красота
2107 Ремонт и отделка
2008 Другое
1997 Республика
1950 Москва
1943 Женщины
1907 Преподаватель
1888 Минимальная сумма заказа
1820 Не выезжаю
1817 Индивидуальные
1580 Предмет или специальность
1478 Подготовка к
1473 Строител

## 6. Собираем текст объявления

Обрабатываем объявления и из корпуса, и из train. Объявления train понадобятся для валидации и для дообучения модели, и обработаны они должны быть **точно так же**, иначе модель будет учиться на одном, а работать на другом.

In [ ]:
DESC_MAX_WORDS = 100

ITEM_COLS = ["item_id", "item_title_raw", "item_description_raw", "item_infm_params_text",
             "item_category_id", "item_microcat_id", "item_location_id",
             "item_latitude", "item_longitude", "item_price", "item_rating",
             "item_rating_reviews_count", "item_is_phone_hidden", "item_is_message_forbidden"]


def process_items(df: pd.DataFrame) -> pd.DataFrame:
    out = df[ITEM_COLS].copy()
    out["title"] = out["item_title_raw"].map(clean_text).str.lower()
    out["desc"] = (out["item_description_raw"].map(clean_text).str.lower()
                   .str.split().str[:DESC_MAX_WORDS].str.join(" "))
    parsed = pd.DataFrame(out["item_infm_params_text"].map(parse_params).tolist(), index=out.index)
    out = pd.concat([out, parsed], axis=1)
    out["item_text"] = (out["title"] + ". " + out["title"] + ". " + out["vid"] + ". " + out["tip"]
                        + ". " + out["params_clean"] + ". " + out["desc"])
    out["item_text"] = out["item_text"].str.replace(r"(\.\s*){2,}", ". ", regex=True).str.strip(". ")
    return out.drop(columns=["item_title_raw", "item_description_raw", "item_infm_params_text"])


items_p = process_items(items)
print("Корпус обработан:", items_p.shape)

# Объявления из train: берём каждое уникальное объявление один раз
train_items_p = process_items(train.drop_duplicates("item_id"))
print("Объявления train обработаны:", train_items_p.shape)

display(items_p[["title", "vid", "tip", "address", "item_text"]].sample(5, random_state=1))

Корпус обработан: (189212, 18)
Объявления train обработаны: (344825, 18)


,title,vid,tip,address,item_text
121061,"оценка авто для нотариуса, наследства уварово",деловые услуги,юридические услуги,"тамбовская обл., уварово","оценка авто для нотариуса, наследства уварово. оценка авто для нотариуса, наследства уварово. деловые услуги. юридические услуги. вид услуги деловые услуги тип услуги юридические услуги направлени..."
183730,доставка чернозема высший сорт,"сад, благоустройство",земляные работы,ул. татарстан,"доставка чернозема высший сорт. доставка чернозема высший сорт. сад, благоустройство. земляные работы. вид услуги сад, благоустройство тип услуги земляные работы работа по договору гарантия бригад..."
80108,"вязаные игрушки ручной работы,крючком и спицами",искусство,,"липецк, площадь ленина-соборная","вязаные игрушки ручной работы,крючком и спицами. вязаные игрушки ручной работы,крючком и спицами. искусство. вид услуги искусство. мелким оптом из наличия,можно повторить на заказ,к сожалению разм..."
86414,грузоперевозки газель next с грузчиками,грузоперевозки,по городу,"москва, улица пришвина, 21",грузоперевозки газель next с грузчиками. грузоперевозки газель next с грузчиками. грузоперевозки. по городу. вид услуги грузоперевозки груз личные вещи груз продукты груз стройматериалы груз техни...
157979,"контент-съемка, мобильный фотограф, визуал, reels",фото- и видеосъемка,,таганская ул.,"контент-съемка, мобильный фотограф, визуал, reels. контент-съемка, мобильный фотограф, визуал, reels. фото- и видеосъемка. вид услуги фото- и видеосъемка работаете с юрлицами и ип как вы работаете..."


### Проверка автоматического разбора

In [ ]:
print(f"Найден вид услуги: {(items_p['vid'] != '').mean():.3f}")
print(f"Найден тип услуги: {(items_p['tip'] != '').mean():.3f}")
print(f"Найден адрес:      {(items_p['address'] != '').mean():.3f}")

addr_len = items_p["address"].str.split().str.len()
print(f"Адресов, обрезанных страховкой (12 слов): {(addr_len >= MAX_ADDRESS_WORDS).mean():.3f}")

print("\nПримеры самых длинных адресов (ищите в них названия полей, которых нет в PARAM_KEYS):")
display(items_p.loc[addr_len >= MAX_ADDRESS_WORDS, "address"].head(10))

print("\nСамые частые виды услуг:")
display(items_p["vid"].value_counts().head(20))

Найден вид услуги: 0.936
Найден тип услуги: 0.731
Найден адрес:      0.997
Адресов, обрезанных страховкой (12 слов): 0.008

Примеры самых длинных адресов (ищите в них названия полей, которых нет в PARAM_KEYS):


31                     московская область, ленинский городской округ, поселок битца, жилой комплекс южная битца, южный
155                  краснодарский край, городской округ сочи, поселок городского типа красная поляна, улица мичурина,
362     ростовская область, мясниковский район, краснокрымское сельское поселение, хутор ленинакан, тупиковая улица, 4
444         чувашская республика, чебоксарский муниципальный округ, поселок городского типа кугеси, советская улица, 1
514                ямало-ненецкий автономный округ, новый уренгой, район южная жилая часть, микрорайон оптимистов, 2/2
596    краснодар, прикубанский внутригородской округ, микрорайон имени петра метальникова, улица цезаря куникова, 24к3
628           москва, юго-восточный административный округ, район марьино, жилой район марьинский парк, 1-й микрорайон
851                   московская область, ленинский городской округ, поселок городского типа дрожжино, новое шоссе, 15
923                     краснодарский край, горо


Самые частые виды услуг:


vid
обучение, курсы                  21375
красота, здоровье                20565
ремонт и отделка                 20063
строительство                    14098
автосервис, аренда               13545
                                 12101
деловые услуги                   12091
праздники, мероприятия            9783
оборудование, производство        9697
ремонт и обслуживание техники     7946
сад, благоустройство              7822
грузоперевозки                    6056
вывоз мусора и вторсырья          4428
фото- и видеосъемка               4043
пассажирские перевозки            3280
другое                            3208
искусство                         3205
компьютерная помощь               2911
бытовые услуги                    2404
уход за животными                 2150
Name: count, dtype: int64

In [ ]:
raw = items["item_infm_params_text"].fillna("")
lost = items_p["vid"].eq("") & raw.str.contains("Вид услуги")
print("Объявлений, где «Вид услуги» есть, а значение пустое:", lost.sum())

# Что стоит сразу после «Вид услуги» в таких объявлениях
after = raw[lost].str.extract(r"Вид услуги (.{0,40})")[0]
display(after.value_counts().head(20))

Объявлений, где «Вид услуги» есть, а значение пустое: 10225


0
Место оказания услуг Московская область,    562
Место оказания услуг Краснодарский край,    316
Место оказания услуг Республика Татарста    292
Место оказания услуг Свердловская област    219
Место оказания услуг Республика Башкорто    155
Место оказания услуг Ставропольский край    119
Место оказания услуг Москва, проезд Воск    100
Место оказания услуг Москва Тип стоимост     97
Место оказания услуг Удмуртская Республи     93
Место оказания услуг Санкт-Петербург, ул     85
Место оказания услуг Ростовская область,     78
Место оказания услуг Чувашская Республик     71
Место оказания услуг Республика Дагестан     61
Место оказания услуг Санкт-Петербург, пр     56
Место оказания услуг Красноярский край,      56
Место оказания услуг Ханты-Мансийский ав     55
Место оказания услуг Свердловская обл.,      52
Место оказания услуг Ленинградская обл.,     49
Место оказания услуг Алтайский край, Бар     49
Место оказания услуг Санкт-Петербург, Дв     45
Name: count, dtype: int64

## 7. Предобработка запросов

In [ ]:
def process_queries(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["q_clean"] = out["search_query"].map(clean_text).str.lower()
    parsed = pd.DataFrame(out["search_infm_params_text"].map(parse_params).tolist(), index=out.index)
    out["search_vid"] = parsed["vid"]
    out["search_tip"] = parsed["tip"]
    out["search_params_clean"] = parsed["params_clean"]
    return out


queries_p = process_queries(queries)
train_q = process_queries(train[Q_COLS + ["item_id"]])

print(f"Benchmark: запросов с фильтром по виду услуги: {(queries_p['search_vid'] != '').mean():.3f}")
print(f"Train:     запросов с фильтром по виду услуги: {(train_q['search_vid'] != '').mean():.3f}")
display(queries_p[["search_query", "q_clean", "search_vid", "search_tip"]].head(10))

Benchmark: запросов с фильтром по виду услуги: 0.340
Train:     запросов с фильтром по виду услуги: 0.644


,search_query,q_clean,search_vid,search_tip
0,перевозки владикавказ тбилиси,перевозки владикавказ тбилиси,,
1,обзвон по базе,обзвон по базе,деловые услуги,
2,липоредукция подбородка,липоредукция подбородка,"красота, здоровье",
3,подъемник 4 х стоечный,подъемник 4 х стоечный,,
4,монтаж видеодомофонов,монтаж видеодомофонов,,
5,боевое самбо,боевое самбо,"обучение, курсы","дизайн, рисование"
6,услуги бухгалтера для ип,услуги бухгалтера для ип,деловые услуги,"бухгалтерия, финансы"
7,бригада по установке забора из профнастила,бригада по установке забора из профнастила,,
8,индивидуальные занятия с тренером,индивидуальные занятия с тренером,,
9,нитрид титана,нитрид титана,"оборудование, производство",


## 7. Сохраняем

Сохраняем четыре файла в `data/processed/`:

Train хранится двумя таблицами, а не одной: одно объявление встречается во многих строках, и незачем хранить его длинный текст много раз. Когда понадобится, склеим по `item_id`.

In [ ]:
items_p.to_parquet(OUT_DIR / "items.parquet", index=False)
train_items_p.to_parquet(OUT_DIR / "train_items.parquet", index=False)
train_q.to_parquet(OUT_DIR / "train_pairs.parquet", index=False)
queries_p.to_parquet(OUT_DIR / "queries.parquet", index=False)

for f in sorted(OUT_DIR.glob("*.parquet")):
    print(f"{f.name:22s} {f.stat().st_size / 1e6:8.1f} MB")

items.parquet             305.7 MB
queries.parquet             0.2 MB
train_items.parquet       543.8 MB
train_pairs.parquet        25.6 MB
